# 07 · Badminton — interaktywne wybrzmienia i opcjonalny mikrofon
**Dane rzeczywiste:** cztery serie central/off-center z jednej rakiety, 46 automatycznych kandydatów.
To demonstracja narzędzi, nie pomiar skuteczności wykrywania wad ceramiki.

Wybierz kernel `.venv` / Ping Lab i **Run All**. Ten notebook działa samodzielnie;
nie musisz wcześniej uruchamiać pozostałych. Granice serii i zapowiedzi pochodzą
z `data/raw/badminton/session.json`. `project.json` pozostaje konfiguracją oryginalnych notebooków.
Nowe wyniki trafiają do osobnego katalogu; nagranie źródłowe pozostaje bez zmian.


In [1]:
NOTEBOOK = '07_nagrywanie_i_wybrzmienia'
from pathlib import Path
import sys, json, hashlib, io, base64
from datetime import datetime
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/pinglab').is_dir()), None)
assert ROOT is not None, 'Otwórz notebook z folderu projektu ceramic-ping-lab.'
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
%matplotlib inline
cfg = json.loads((ROOT/'data/raw/badminton/session.json').read_text())
SOURCE = ROOT/'data/raw/badminton'/cfg['audio']
assert SOURCE.is_file(), 'Brak nagrania badminton: nie przełączamy się automatycznie na demo.'
OUT = ROOT/'reports/runs/badminton-notebooks'/NOTEBOOK/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Audio, HTML, Markdown
from pinglab.audio import load_audio, Config
from pinglab.recorder import segment_ringdown, ringdown_plot
from pinglab.features import extract_features, feature_columns, decay_curve
from pinglab.learning import make_classifier
from pinglab.series import evaluate_series, paired_folds
from pinglab.embeddings import ASTEncoder
from pinglab.robustness import mix_noise

RUN_AST = False
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})
colors = {'central':'#137f83', 'offcenter':'#c56838'}
r = load_audio(SOURCE, Config(sr=48000, max_seconds=180))
print('Plik:', SOURCE.name, '| SHA256:', hashlib.sha256(SOURCE.read_bytes()).hexdigest())
display(pd.DataFrame([r.qc]).drop(columns=['warnings']))
print('Uwagi QC:', r.qc['warnings'] or 'brak')

Plik: session.wav | SHA256: afc2d308b32b80ead3f80f795707a2059d2e359a33a2d6c20470464d41df7f07


,duration_s,native_sr,channels,selected_channel,peak_dbfs,rms_dbfs,clipping_fraction,dc_offset
0,88.99771,44100,1,0,-17.976912,-45.595115,0.0,8.182004e-09


Uwagi QC: ['Pasmo źródłowe kończy się przy 22050 Hz; resampling go nie odzyska']


## Wybierz serię lub nowe nagranie
Domyślnie analizujemy central_1 z istniejącego nagrania. Mikrofon jest używany wyłącznie po zmianie `RECORD=True`. Cała sesja zawiera zapowiedzi głosowe, dlatego detektor dostaje wybraną serię bez mowy.

In [2]:
from pinglab.recorder import record_wav, export_pings
RECORD=False
SERIES_ID='central_1'
SECONDS=20
DEVICE=None
if RECORD:
 source_path=record_wav(ROOT/'data/raw/recordings',SECONDS,DEVICE,None)
 recorded=load_audio(source_path,Config(max_seconds=180))
 y,sr=recorded.y,recorded.sr
else:
 selected=next(s for s in cfg['series'] if s['series_id']==SERIES_ID)
 y=r.y[int(selected['start_s']*r.sr):int(selected['end_s']*r.sr)];sr=r.sr
print('Nowe nagranie' if RECORD else SERIES_ID)
display(Audio(y,rate=sr,normalize=False))
MANUAL_ONSETS=None
MIN_GAP=.4
SENSITIVITY=.15
DROP_DB=30
QUIET_HOLD=.08


central_1


## Uderzenia i wybrzmienia
Czerwony: umowne pierwsze 30 ms. Zielony: obserwowany zanik. Pomarańczowy: brak zaniku przed następnym zdarzeniem lub końcem pliku. To heurystyka; szum też wpływa na granicę. Zaznacz myszą fragment wykresu, aby go przybliżyć.

In [3]:
segments,t,db=segment_ringdown(y,sr,MANUAL_ONSETS,min_gap_s=MIN_GAP,sensitivity=SENSITIVITY,drop_db=DROP_DB,quiet_hold_s=QUIET_HOLD)
fig=ringdown_plot(y,sr,segments,t,db)
fig.show(renderer='plotly_mimetype')
fig.write_html(OUT/'wybrzmienia.html',include_plotlyjs=True)
display(segments.round(3))
paths=export_pings(y,sr,segments,OUT/'pings')
print('Wyeksportowane fragmenty:',len(paths))
for row in list(segments.itertuples())[:12]:
 print('Ping',row.ping,row.status)
 display(Audio(y[max(0,int((row.onset_s-.015)*sr)):int(row.end_s*sr)],rate=sr,normalize=False))


,ping,onset_s,attack_end_s,end_s,duration_s,status,threshold_dbfs,noise_dbfs,onset_in_noise_window
0,1,2.025,2.055,2.095,0.070,zanik zaobserwowany,-58.359,-64.359,False
1,2,3.170,3.200,3.270,0.100,zanik zaobserwowany,-58.359,-64.359,False
2,3,4.360,4.390,4.500,0.140,zanik zaobserwowany,-58.359,-64.359,False
3,4,5.385,5.415,5.700,0.315,zanik zaobserwowany,-58.359,-64.359,False
4,5,6.445,6.475,6.615,0.170,zanik zaobserwowany,-58.359,-64.359,False
5,6,7.610,7.640,8.665,1.055,kolejne uderzenie przed zanikiem,-58.359,-64.359,False
6,7,8.665,8.695,9.235,0.570,zanik zaobserwowany,-58.359,-64.359,False
7,8,9.800,9.830,10.015,0.215,zanik zaobserwowany,-58.359,-64.359,False
8,9,10.885,10.915,11.005,0.120,zanik zaobserwowany,-58.359,-64.359,False
9,10,12.110,12.140,12.225,0.115,zanik zaobserwowany,-58.359,-64.359,False


Wyeksportowane fragmenty: 14
Ping 1 zanik zaobserwowany


Ping 2 zanik zaobserwowany


Ping 3 zanik zaobserwowany


Ping 4 zanik zaobserwowany


Ping 5 zanik zaobserwowany


... (3 outputs left out)

Ping 9 zanik zaobserwowany


Ping 10 zanik zaobserwowany


Ping 11 zanik zaobserwowany


Ping 12 zanik zaobserwowany


In [4]:
print('GOTOWE:', NOTEBOOK, '| pliki wynikowe:', OUT)

GOTOWE: 07_nagrywanie_i_wybrzmienia | pliki wynikowe: /Users/karolsii/dev/ceramic-ping-lab/reports/runs/badminton-notebooks/07_nagrywanie_i_wybrzmienia/20260928_144921_653481
